In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

In [2]:
df_epi_raw = pd.read_csv("data/epi_r.csv", sep=",")
df_epi_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 104.0 MB


## Duplicates


In [3]:
df_epi_raw.duplicated().sum()

np.int64(1801)

In [4]:
df_epi_drop_dup = df_epi_raw.drop_duplicates()

## DownCast 

In [5]:
(df_epi_drop_dup.nunique()).value_counts()

2        674
17736      1
8          1
1858       1
282        1
326        1
2434       1
Name: count, dtype: int64

Т.е. 674 столбцы имеют 2 униткальных значения, 1 столбец - 17736, 1 - 8 уникальных значений.

Найдем столбцы с 2 уникальными значениями.

In [6]:
mask_cols_eq2 = df_epi_drop_dup.nunique() == 2
cols_eq2 = df_epi_drop_dup.columns[mask_cols_eq2]

In [7]:
df_epi_drop_dup.columns[~mask_cols_eq2]

Index(['title', 'rating', 'calories', 'protein', 'fat', 'sodium'], dtype='str')

In [8]:
df_epi_drop_dup.info(memory_usage=True)

<class 'pandas.DataFrame'>
Index: 18251 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 94.8 MB


In [9]:
df_epi = df_epi_drop_dup.copy()

In [10]:
for column in df_epi_drop_dup.columns[mask_cols_eq2]:
    df_epi[column] = pd.to_numeric(df_epi[column], downcast="integer")

In [11]:
df_epi.info(memory_usage=True)

<class 'pandas.DataFrame'>
Index: 18251 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(5), int8(674), str(1)
memory usage: 12.7 MB


## Columns filter

Check:
    - frequencies (kiwi not pass)
    - correlation with other columns
    - parcers (ingredient_slicer, pyfood)

In [12]:
import pandas as pd
import re
from recipe import StrictIngredientSelector

all_columns_eq2 = list(df_epi.columns[mask_cols_eq2])

selector = StrictIngredientSelector()
final_ingredients = selector.filter_columns(all_columns_eq2)

print(f"Осталось чистых ингредиентов: {len(final_ingredients)}")
print("Первые элементы списка:", final_ingredients[:15])

Осталось чистых ингредиентов: 345
Первые элементы списка: ['almond', 'amaretto', 'anchovy', 'anise', 'apple', 'apple juice', 'apricot', 'artichoke', 'arugula', 'asian pear', 'asparagus', 'avocado', 'bacon', 'banana', 'barley']


In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
from recipe import DataCleaner, FeatureExtractor


from sklearn.pipeline import Pipeline

preprocessing_pipe = Pipeline(
    [("data_cleaner", DataCleaner()), ("feature_extractor", FeatureExtractor())]
)

preprocessing_pipe.fit_transform(df_epi_raw)

[DataCleaner] After drop duplicates. Shape = (20052, 680)
[DataCleaner] After drop duplicates. Shape = (18235, 680)
[DataCleaner] Before memory optimization
<class 'pandas.DataFrame'>
Index: 18235 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 94.7 MB
None

[DataCleaner] After memory optimization
<class 'pandas.DataFrame'>
Index: 18235 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(5), int8(674), str(1)
memory usage: 12.7 MB
None

[DataCleaner] Final ingredients count: 345
[FeatureExtractor] Feature extracted. (18235, 346)


,rating,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,...,watercress,watermelon,whiskey,white wine,wild rice,wine,yogurt,yuca,zucchini,turkey
0,2.500,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,4.375,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3.750,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5.000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,3.125,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20047,3.125,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
20048,4.375,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
20049,4.375,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
20050,4.375,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
def titles_by_ingredients(col, df):
    return df[df[col] == 1][["title", "rating"]]


titles_by_ingredients("spirit", df_epi)

,title,rating
239,Apple Treacle Tart,4.375
420,Amaretto Olive Oil Cake,3.750
547,Blue Mountain,3.750
576,Orange Gin Fizz,0.000
769,Moulin Rouge,0.000
...,...,...
19861,Alaska,0.000
19924,Tipsy Tea with Homemade Sweet Tea,5.000
19938,Scotch Cooler,0.000
20000,Self-Starter,0.000


In [15]:
assert 1 == 0

AssertionError: 